Plots: Point Source
===================

This example shows how to plot a `PointDataset` and a `FitPointDataset` fit.

Point-source data differs from imaging and interferometer data: it consists of the (y,x) positions
of a point source's multiple images (and optionally their fluxes and time delays), not a 2D image.
Plotting therefore centres on the multiple-image positions and the fit's ability to trace them back
to a common source.

For an introduction to the plotting API refer to `guides/plot/start_here.py`. The final section
documents the `Visualizer`, which outputs point-source fit figures automatically during a model-fit.

__Contents__

- **Dataset:** Load the point-source dataset of multiple-image positions.
- **Dataset Figures:** Inspect the dataset's info and plot its multiple-image positions.
- **Dataset Subplot:** Plot all dataset quantities in one multi-panel subplot.
- **Fit:** Set up a tracer and point solver and fit the dataset with a `FitPointDataset`.
- **Fit Figures:** Inspect the fit's residuals and plot the fit subplot.
- **Visualizer:** How these figures are output automatically during a model-fit.

__Google Colab Setup__

This cell sets up the environment when the notebook is run on Google Colab: it installs the
required PyAuto packages, clones the workspace (configuration files and example datasets) and
points the configuration at it. If you are running the notebook elsewhere (e.g. locally via
your own installation) it does nothing, and you can run it safely.

Colab tip: model-fits run much faster on a GPU — enable one via "Runtime" -> "Change runtime
type" -> "Hardware accelerator" before running the notebook.

In [ ]:
try:
    import google.colab
except ImportError:
    from autolens import setup_colab as _setup_colab
else:
    import importlib
    import subprocess
    import sys

    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "autonerves", "--no-deps"]
    )
    _setup_colab = importlib.import_module("autonerves.setup_colab")

_setup_colab.setup("autolens")

In [ ]:

from autolens import jax_wrapper  # Sets JAX environment before other imports

# from autolens import setup_notebook; setup_notebook()

from pathlib import Path
import autolens as al
import autolens.plot as aplt

__Dataset__

Load the point-source dataset `simple`, which contains the positions of a lensed quasar's multiple
images.

In [ ]:
dataset_name = "simple"
dataset_path = Path("dataset") / "point_source" / dataset_name

__Dataset Auto-Simulation__

If the dataset does not already exist on your system, it will be created by running the corresponding
simulator script. This ensures that all example scripts can be run without manually simulating data first.

In [ ]:
if not (dataset_path / "point_dataset_positions_only.json").exists():
    import subprocess
    import sys

    subprocess.run(
        [sys.executable, "scripts/point_source/simulator.py"],
        check=True,
    )

dataset = al.from_json(
    file_path=Path(dataset_path, "point_dataset_positions_only.json"),
)

__Dataset Figures__

The dataset's `info` summarizes its name, positions and noise-map values. The name (`point_0`) is
what pairs the dataset to a point source in a lens model (see `scripts/point_source/fit.py`).

In [ ]:
print(dataset.info)

The multiple-image positions are a `Grid2DIrregular`, plotted with `aplt.plot_grid()`.

In [ ]:
aplt.plot_grid(grid=dataset.positions, title="Multiple Image Positions")

__Dataset Subplot__

A multi-panel subplot of the dataset is produced with `aplt.subplot_point_dataset()`, combining the
multiple-image positions with the dataset's other quantities (its noise-map, and its fluxes and
time-delays where these are measured).

In [ ]:
aplt.subplot_point_dataset(dataset=dataset)

__Fit__

To fit the positions we compose a tracer whose mass model and point source match the true simulated
values, and a `PointSolver` which solves the lens equation to find where the point source's multiple
images appear.

In [ ]:
lens_galaxy = al.Galaxy(
    redshift=0.5,
    mass=al.mp.Isothermal(
        centre=(0.0, 0.0),
        einstein_radius=1.8,
        ell_comps=al.convert.ell_comps_from(axis_ratio=0.9, angle=45.0),
    ),
)

source_galaxy = al.Galaxy(
    redshift=1.0, point_0=al.ps.PointFlux(centre=(0.0, 0.0), flux=0.8)
)

tracer = al.Tracer(galaxies=[lens_galaxy, source_galaxy])

point_grid = al.Grid2D.uniform(
    shape_native=(100, 100),
    pixel_scales=0.2,
)

solver = al.PointSolver.for_grid(
    grid=point_grid, pixel_scale_precision=0.001, magnification_threshold=0.1
)

fit = al.FitPointDataset(dataset=dataset, tracer=tracer, solver=solver)

__Fit Figures__

The fit's positions component contains the residuals between the observed and model multiple-image
positions, which are irregular arrays inspected by printing.

In [ ]:
print(fit.positions.residual_map)
print(fit.positions.normalized_residual_map)
print(fit.positions.chi_squared_map)

The fit subplot plots the observed and model positions together, the primary figure for checking a
point-source fit.

In [ ]:
aplt.subplot_fit_point(fit=fit)

__Visualizer__

During a model-fit (e.g. `search.fit(model=model, analysis=analysis)` in `modeling.py`), the fit
figures above are output to hard-disk automatically by the `Visualizer` attached to the `Analysis`
class:

In [ ]:
print(al.AnalysisPoint.Visualizer)

At regular intervals during the non-linear search, and again once it finishes, the `Visualizer`
outputs the maximum likelihood fit's figures to the fit's output folder, under `image/`
(e.g. `output/<path_prefix>/<name>/image/`).

Which figures are output is controlled by the config file `config/visualize/plots.yaml`. The
machinery is described in full in `scripts/imaging/plot.py` — the same config-driven visualization
applies to every dataset type.